%pip install optbinning pandas xlrd duckdb
%pip install --upgrade pip


In [38]:
import pandas as pd
from itertools import combinations
from optbinning import OptimalBinning


In [39]:
data = pd.read_csv(r"/workspaces/Strategic_Segment_Builder/clean_dataset.csv")
data = data.drop(columns=["ZipCode"])
data = pd.get_dummies(data, columns=['Industry','Ethnicity','Citizen'], drop_first= True,dtype=int )
target = "Approved"
data.head()


,Gender,Age,Debt,Married,BankCustomer,YearsEmployed,PriorDefault,Employed,CreditScore,DriversLicense,...,Industry_Real Estate,Industry_Research,Industry_Transport,Industry_Utilities,Ethnicity_Black,Ethnicity_Latino,Ethnicity_Other,Ethnicity_White,Citizen_ByOtherMeans,Citizen_Temporary
0,1,30.83,0.000,1,1,1.25,1,1,1,0,...,0,0,0,0,0,0,0,1,0,0
1,0,58.67,4.460,1,1,3.04,1,1,6,0,...,0,0,0,0,1,0,0,0,0,0
2,0,24.50,0.500,1,1,1.50,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,1,27.83,1.540,1,1,3.75,1,1,5,1,...,0,0,0,0,0,0,0,1,0,0
4,1,20.17,5.625,1,1,1.71,1,0,0,0,...,0,0,0,0,0,0,0,1,1,0


In [40]:
data['Approved'].value_counts(normalize=True)*100

Approved
0    55.507246
1    44.492754
Name: proportion, dtype: float64

In [41]:
def target_value_counts(df, target_col, normalize=True, pct=True):
    counts = df[target_col].value_counts(normalize=normalize)
    return counts * 100 if pct else counts

def compute_iv_ranking(df, target, solver="cp"):
    results = []
    for col in df.columns:
        if col == target:
            continue
        try:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category","str"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype, solver=solver)
            optb.fit(df[col].values, df[target].values)
            iv = optb.binning_table.build().IV.iloc[-1] 
            results.append({"variable": col, "iv": iv * 100})
        except Exception as exc:
            print(f"Skipping {col}: {exc}")
    return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

def create_binned_df(df, target, variables):
    binned_df = pd.DataFrame(index=df.index)
    for col in variables:
        dtype = "categorical" if str(df[col].dtype) in ["object", "category","str"] else "numerical"
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(df[col].values, df[target].values)
        binned_df[col] = optb.transform(df[col], metric="bins")
    binned_df[target] = df[target].values
    return binned_df


In [42]:
def one_way_summary(binned_df, target, base_rate, pct=True):
    results = []
    for col in binned_df.columns:
        if col == target:
            continue
        summary = (
            binned_df
            .groupby(col)
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = col + "=" + summary[col].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def two_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2 in combinations(cols, 2):
        summary = (
            binned_df
            .groupby([c1, c2])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = c1 + "=" + summary[c1].astype(str) + sep + c2 + "=" + summary[c2].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def three_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2, c3 in combinations(cols, 3):
        summary = (
            binned_df
            .groupby([c1, c2, c3])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = (
            c1 + "=" + summary[c1].astype(str) + sep +
            c2 + "=" + summary[c2].astype(str) + sep +
            c3 + "=" + summary[c3].astype(str)
        )
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def shortlist_rules(df, min_sample_size=1000, min_lift=2.0, base_rate=None):
    if base_rate is None:
        raise ValueError("base_rate is required")
    threshold = base_rate * 100
    return df.assign(
        shortlist=(
            (df["count"] >= min_sample_size) &
            (df["lift"] >= min_lift) &
            (df["rate"] > threshold)
        ).astype(int)
    )


In [43]:
data.shape

(690, 31)

In [44]:
iv_ranking = compute_iv_ranking(data, target)
top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(data, target, top_vars)
base_rate = data[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=100, min_lift=2, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=100, min_lift=2, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=100, min_lift=1.5, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",119,92.436975,2.077574,1
1,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",185,92.432432,2.077472,1
2,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",193,92.227979,2.072876,1
3,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",193,92.227979,2.072876,1
4,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",217,92.165899,2.071481,1
5,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",128,91.406250,2.054408,1
6,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",221,91.402715,2.054328,1
7,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",220,91.363636,2.053450,1
8,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",217,91.244240,2.050766,1
9,"PriorDefault=[0.50, inf) & Employed=[0.50, inf)",228,90.789474,2.040545,1


In [45]:
def parse_rule(rule_str):
    parts = [p.strip() for p in rule_str.split("&")]
    conditions = []
    for part in parts:
        column, interval = part.split("=", 1)
        column = column.strip()
        interval = interval.strip()
        include_lower = interval.startswith("[")
        include_upper = interval.endswith("]")
        lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
        def to_bound(value):
            value = value.lower()
            if value in {"inf", "infty", "+inf", "+infty"}:
                return float("inf")
            if value in {"-inf", "-infty"}:
                return float("-inf")
            return float(value)
        conditions.append({
            "column": column,
            "lower": to_bound(lower_str),
            "upper": to_bound(upper_str),
            "include_lower": include_lower,
            "include_upper": include_upper,
        })
    return conditions

def build_mask(df, conditions):
    mask = pd.Series(True, index=df.index)
    for cond in conditions:
        column = cond["column"]
        column_values = pd.to_numeric(df[column], errors="coerce")
        current = pd.Series(True, index=df.index)
        if cond["include_lower"]:
            current &= column_values >= cond["lower"]
        else:
            current &= column_values > cond["lower"]
        if cond["include_upper"]:
            current &= column_values <= cond["upper"]
        else:
            current &= column_values < cond["upper"]
        mask &= current.fillna(False)
    return mask

first_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(first_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100


Approved
0    65.499124
1    34.500876
Name: proportion, dtype: float64

In [46]:
df_filtered.shape

(571, 31)

In [47]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=100, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=100, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=100, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",104,91.346154,2.647647,1
1,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",104,90.384615,2.619777,1
2,"PriorDefault=[0.50, inf) & Employed=[0.50, inf)",109,88.990826,2.579379,1
3,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",109,88.990826,2.579379,1
4,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",109,88.990826,2.579379,1
5,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",109,88.990826,2.579379,1
6,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",106,88.679245,2.570348,1
7,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",106,88.679245,2.570348,1
8,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",105,88.571429,2.567223,1
9,"PriorDefault=[0.50, inf) & BankCustomer=[0.50,...",173,79.190751,2.295326,1


In [48]:
second_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(second_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Approved
0    78.158458
1    21.841542
Name: proportion, dtype: float64

In [49]:
fifth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(fifth_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Approved
0    78.158458
1    21.841542
Name: proportion, dtype: float64

In [50]:
import numpy as np

In [51]:
def convert_interval_to_query(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a pandas .query() compatible string.
    """
    query_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column
        if col_conditions:
            query_conditions.append(" & ".join(col_conditions))
            
    # Combine all column rules into the final query string
    return " & ".join(query_conditions)

# --- Test the converter ---
input_string = first_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PriorDefault=[0.50, inf) & Employed=[0.50, inf) & DriversLicense=(-inf, 0.50)
Output: PriorDefault >= 0.50 & Employed >= 0.50 & DriversLicense < 0.50


In [52]:
# --- Test the converter ---
input_string = second_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PriorDefault=[0.50, inf) & Employed=[0.50, inf) & Industry_Financials=(-inf, 0.50)
Output: PriorDefault >= 0.50 & Employed >= 0.50 & Industry_Financials < 0.50


In [53]:
# --- Test the converter ---
input_string = third_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PriorDefault=[0.50, inf) & Citizen_ByOtherMeans=(-inf, 0.50) & Industry_ConsumerStaples=(-inf, 0.50)
Output: PriorDefault >= 0.50 & Citizen_ByOtherMeans < 0.50 & Industry_ConsumerStaples < 0.50


In [54]:
def convert_interval_to_sql(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a SQL WHERE clause compatible string.
    """
    sql_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column using SQL 'AND'
        if col_conditions:
            sql_conditions.append(" AND ".join(col_conditions))
            
    # Combine all column rules into the final SQL string using 'AND'
    # Parentheses are added around variables that have both an upper and lower bound
    return " AND ".join(f"({cond})" if "AND" in cond else cond for cond in sql_conditions)

# --- Test the converter ---
input_string = first_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PriorDefault=[0.50, inf) & Employed=[0.50, inf) & DriversLicense=(-inf, 0.50)
Output SQL WHERE clause: PriorDefault >= 0.50 AND Employed >= 0.50 AND DriversLicense < 0.50


In [55]:
# --- Test the converter ---
input_string = second_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PriorDefault=[0.50, inf) & Employed=[0.50, inf) & Industry_Financials=(-inf, 0.50)
Output SQL WHERE clause: PriorDefault >= 0.50 AND Employed >= 0.50 AND Industry_Financials < 0.50


In [56]:
# --- Test the converter ---
input_string = third_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PriorDefault=[0.50, inf) & Citizen_ByOtherMeans=(-inf, 0.50) & Industry_ConsumerStaples=(-inf, 0.50)
Output SQL WHERE clause: PriorDefault >= 0.50 AND Citizen_ByOtherMeans < 0.50 AND Industry_ConsumerStaples < 0.50


In [58]:
import duckdb
result = duckdb.sql("""

SELECT CASE 
WHEN PriorDefault >= 0.50 AND Employed >= 0.50 AND DriversLicense < 0.50 THEN 1
WHEN PriorDefault >= 0.50 AND Employed >= 0.50 AND Industry_Financials < 0.50 THEN 2
WHEN PriorDefault >= 0.50 AND Citizen_ByOtherMeans < 0.50 AND Industry_ConsumerStaples < 0.50 THEN 3
ELSE 0 END AS SEGMENTS, 
COUNT(*) AS COUNTS,
SUM(Approved) AS EVENT,
(SUM(Approved) / COUNT(*)) * 100 AS RESPONSE_RATE,
FROM data
GROUP BY SEGMENTS

""").df()

result



,SEGMENTS,COUNTS,EVENT,RESPONSE_RATE
0,0,365,37.0,10.136986
1,1,119,110.0,92.436975
2,2,104,95.0,91.346154
3,3,102,65.0,63.725490
